# Solution A1 · Bagging from scratch: bootstrap, vote and out-of-bag error

**Type**: Implement from scratch

## Objective

Implement bagging of trees with numpy and scikit-learn (only `DecisionTreeClassifier` as a building block): bootstrap samples, majority vote and **out-of-bag error** computed by hand, checking that it matches the estimate of `BaggingClassifier`. Also measure that a bootstrap sample contains 63.2% of the distinct examples.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)

## Commented solution

In [ ]:
def bagging(X, y, B, seed=0):
    rng = np.random.default_rng(seed)
    trees, indices = [], []
    for b in range(B):
        i = rng.integers(0, len(X), len(X))                                  # bootstrap sample: with replacement
        trees.append(DecisionTreeClassifier(random_state=b).fit(X[i], y[i]))
        indices.append(i)
    return trees, indices


def vote_predict(trees, X):
    return (np.mean([t.predict(X) for t in trees], axis=0) > 0.5).astype(int)


def oob(trees, indices, X, y):
    n = len(X)
    votes, n_votes = np.zeros(n), np.zeros(n)
    for tree, i in zip(trees, indices):
        outside = np.setdiff1d(np.arange(n), i)                              # examples this tree has not seen
        votes[outside] += tree.predict(X[outside])
        n_votes[outside] += 1
    covered = n_votes > 0
    predictions = np.where(covered, votes / np.maximum(n_votes, 1) > 0.5, 0).astype(int)
    return predictions, covered

In [ ]:
trees, indices = bagging(X_tr, y_tr, 100)
single_acc = float(np.mean([t.score(X_te, y_te) for t in trees]))
vote_acc = float((vote_predict(trees, X_te) == y_te).mean())
distinct_share = float(np.mean([len(np.unique(i)) / len(y_tr) for i in indices]))
print(f"mean accuracy of a single tree: {single_acc:.3f} | vote of 100 trees: {vote_acc:.3f} | share of distinct examples in a bootstrap: {distinct_share:.3f}")

In [ ]:
oob_predictions, covered = oob(trees, indices, X_tr, y_tr)
oob_acc = float((oob_predictions[covered] == y_tr[covered]).mean())
covered_share = float(covered.mean())
sklearn_acc = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=0).fit(X_tr, y_tr).oob_score_
print(f"OOB accuracy (by hand): {oob_acc:.3f} | BaggingClassifier: {sklearn_acc:.3f} | test accuracy of the vote: {vote_acc:.3f} | covered examples: {covered_share:.1%}")

## Check

In [ ]:
with verify():
    check(len(trees) == 100 and len(indices) == 100 and all(len(i) == len(y_tr) for i in indices), "100 trees and 100 bootstrap samples of the same size as the training set", "rng.integers(0, n, n)")
    check(0.62 < distinct_share < 0.645, "a bootstrap contains on average 63.2% of distinct examples", "len(np.unique(i)) / n")
    check(vote_acc > single_acc + 0.06 and single_acc < 0.8 and vote_acc > 0.82, "the vote (over 0.82) beats the single tree (below 0.8) by over 6 points", "mean of the predictions > 0.5")
    check(covered_share > 0.99 and oob_predictions.shape == y_tr.shape, "with 100 trees almost all the examples have at least one OOB tree", "an example is covered if it is missing from the bootstrap of at least one tree")
    check(abs(oob_acc - vote_acc) < 0.05 and abs(oob_acc - sklearn_acc) < 0.02, "the OOB accuracy is close to the test one (within 5 points) and to the estimate of BaggingClassifier (within 2)", "only trees that have not seen the example")
    check(oob_acc < 0.999, "the OOB is not 1.0: it is not the estimate on the training set (which is almost perfect)", "an example never votes with the trees that have seen it")
done()

## What you learned

- **Bagging is a three-line loop:** sample with replacement, tree, vote. A single tree without limits gets 0.775; the vote of 100 trees 0.845: 7 points gained **without changing the tree algorithm**, just by averaging its variance.
- **A bootstrap contains 63.2% of distinct examples** ($1 - 1/e$): the remaining 36.8% is the out-of-bag set of that tree. With 100 trees the probability that an example is "inside" in all of them is tiny, so almost all of them receive at least one OOB prediction.
- **The OOB error is a free honest estimate:** 0.862 against 0.845 on the test set, much closer to reality than the training accuracy (almost 1.0). Every OOB prediction uses only the trees that have never seen that example, so it is a piece of validation built into the training.
- The random forest adds only one thing to bagging: at **every split** the tree can choose only among a random subset of features. It serves to decorrelate the trees (A2).

## Common pitfall

Computing the accuracy of a forest or of a bagging on the **training set** and reporting it: it is almost 1.0 because every example has been seen by about 63% of the trees. The estimate to report is the OOB or the one on a test set.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What is the out-of-bag error and when can you trust it?</summary>

Each tree is trained on a bootstrap that leaves out about 37% of the examples: predicting each example only with the trees that have not seen it gives me an honest prediction for the whole training set, and I compute its error, without a validation set. It is a good estimate when the examples are
independent; with duplicate or correlated rows (same customer, time series) it is optimistic and you need a validation by group or by time.

</details>